In [1]:
import pandas as pd
import torch
import numpy as np
import os
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

print("\n1. Mounting Google Drive...")
drive.mount('/content/drive')
print("Drive mounted")

model_path = '/content/drive/MyDrive/distilbert_toxicity_model_final'

if os.path.exists(model_path):
    print(f"✓ Model found at: {model_path}")
else:
    print(f"❌ Model not found at: {model_path}")
    print("   Checking alternative locations...")

    # Alternative: check if model is in current directory
    if os.path.exists('./distilbert_toxicity_model_final'):
        model_path = './distilbert_toxicity_model_final'
        print(f"✓ Model found locally at: {model_path}")
    else:
        print("   Please ensure model is saved in Drive")

print("\n2. Loading saved data...")

if os.path.exists('train_subset_cleaned.csv') and os.path.exists('eval_subset_cleaned.csv'):
    train_df = pd.read_csv('train_subset_cleaned.csv')
    eval_df = pd.read_csv('eval_subset_cleaned.csv')
    print("   Loaded from current directory")
else:
    train_df = pd.read_csv('/content/drive/MyDrive/train_subset_cleaned.csv')
    eval_df = pd.read_csv('/content/drive/MyDrive/eval_subset_cleaned.csv')
    print("   Loaded from Google Drive")

print(f"   Training set: {len(train_df):,} rows")
print(f"   Evaluation set: {len(eval_df):,} rows")
print(f"   Training toxic %: {train_df['toxic_binary'].mean():.2%}")
print(f"   Evaluation toxic %: {eval_df['toxic_binary'].mean():.2%}")

print("\n3. Loading saved model from Drive...")
model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()
print(f"   Model loaded on {device}")
print(f"   Model parameters: {model.num_parameters():,}")

print("\n4. Verification:")
print(f"   Eval toxic %: {eval_df['toxic_binary'].mean():.2%}")
print(f"   Model device: {next(model.parameters()).device}")

print("\nSetup complete! Ready for analysis.")


1. Mounting Google Drive...
Mounted at /content/drive
Drive mounted
✓ Model found at: /content/drive/MyDrive/distilbert_toxicity_model_final

2. Loading saved data...
   Loaded from current directory
   Training set: 100,000 rows
   Evaluation set: 20,000 rows
   Training toxic %: 8.00%
   Evaluation toxic %: 8.00%

3. Loading saved model from Drive...


Loading weights:   0%|          | 0/104 [00:01<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

   Model loaded on cuda
   Model parameters: 66,955,010

4. Verification:
   Eval toxic %: 8.00%
   Model device: cuda:0

Setup complete! Ready for analysis.


# Part 3: Adversarial Attacks

In [2]:
print("PART 3: ADVERSARIAL ATTACKS (FIXED)")
import random
import re
from tqdm import tqdm

print("ATTACK 1: CHARACTER-LEVEL EVASION ATTACK")

def zero_width_insertion(text):
    zero_width = '\u200B'
    toxic_words = ['hate', 'kill', 'stupid', 'idiot', 'attack', 'hurt', 'die', 'loser', 'suck', 'trash']

    result = text
    for word in toxic_words:
        if word.lower() in result.lower():
            pattern = re.compile(re.escape(word), re.IGNORECASE)
            def replace_match(match):
                original = match.group(0)
                new_word = ''
                for i, char in enumerate(original):
                    new_word += char
                    if i > 0 and i % random.randint(2, 3) == 0 and i < len(original) - 1:
                        new_word += zero_width
                return new_word
            result = pattern.sub(replace_match, result)
    return result

def homoglyph_substitution(text):
    homoglyphs = {
        'a': '\u0430', 'e': '\u0435', 'o': '\u043e', 'p': '\u0440', 'c': '\u0441', 'x': '\u0445', 'y': '\u0443',
    }
    result = text
    for latin, cyrillic in homoglyphs.items():
        result = result.replace(latin, cyrillic)
        result = result.replace(latin.upper(), cyrillic.upper())
    return result

def random_duplication(text):
    words = text.split()
    new_words = []
    for word in words:
        if len(word) < 2:
            new_words.append(word)
            continue
        chars = list(word)
        new_chars = []
        for i, char in enumerate(chars):
            new_chars.append(char)
            if random.random() < 0.2:
                new_chars.append(char)
        new_words.append(''.join(new_chars))
    return ' '.join(new_words)

def perturb(text):
    text = zero_width_insertion(text)
    text = homoglyph_substitution(text)
    text = random_duplication(text)
    return text

sample_toxic = "I hate you, you're so stupid and ugly"
print(f"\n1. Testing attack on sample text:")
print(f"   Original: {sample_toxic}")
print(f"   Perturbed: {perturb(sample_toxic)}")

print("\n2. Finding toxic comments with confidence >= 0.7...")

batch_size = 500
toxic_indices = []
toxic_probs = []

model.eval()
for i in tqdm(range(0, len(eval_df), batch_size), desc="Finding toxic comments"):
    batch_texts = eval_df['comment_text'].iloc[i:i+batch_size].tolist()
    batch_labels = eval_df['toxic_binary'].iloc[i:i+batch_size].values

    if sum(batch_labels) > 0:
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
        inputs = {k: v.to('cuda') for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()

        for j, (prob, label) in enumerate(zip(probs, batch_labels)):
            if label == 1 and prob >= 0.7:
                toxic_indices.append(i + j)
                toxic_probs.append(prob)

print(f"   Found {len(toxic_indices)} toxic comments with confidence >= 0.7")

sample_size = min(500, len(toxic_indices))
if sample_size == 0:
    print("   No high-confidence toxic comments found. Using lower threshold...")
    toxic_indices = []
    for i in tqdm(range(0, len(eval_df), batch_size), desc="Finding toxic comments (threshold 0.5)"):
        batch_texts = eval_df['comment_text'].iloc[i:i+batch_size].tolist()
        batch_labels = eval_df['toxic_binary'].iloc[i:i+batch_size].values

        if sum(batch_labels) > 0:
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors='pt')
            inputs = {k: v.to('cuda') for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model(**inputs)
                probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()

            for j, (prob, label) in enumerate(zip(probs, batch_labels)):
                if label == 1 and prob >= 0.5:
                    toxic_indices.append(i + j)
    sample_size = min(500, len(toxic_indices))

attack_samples = random.sample(toxic_indices, sample_size) if sample_size > 0 else []
print(f"   Testing on {sample_size} samples...")

original_confidences = []
perturbed_confidences = []
original_predictions = []
perturbed_predictions = []

for idx in tqdm(attack_samples, desc="Running evasion attack"):
    original_text = eval_df.loc[idx, 'comment_text']

    inputs = tokenizer(original_text, return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to('cuda') for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        original_prob = torch.softmax(outputs.logits, dim=-1)[0, 1].item()
    original_pred = 1 if original_prob >= 0.6 else 0

    perturbed_text = perturb(original_text)

    inputs = tokenizer(perturbed_text, return_tensors='pt', truncation=True, max_length=128)
    inputs = {k: v.to('cuda') for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        perturbed_prob = torch.softmax(outputs.logits, dim=-1)[0, 1].item()
    perturbed_pred = 1 if perturbed_prob >= 0.6 else 0

    original_confidences.append(original_prob)
    perturbed_confidences.append(perturbed_prob)
    original_predictions.append(original_pred)
    perturbed_predictions.append(perturbed_pred)

if len(attack_samples) > 0:
    successful_evasions = sum(1 for orig, pert in zip(original_predictions, perturbed_predictions)
                              if orig == 1 and pert == 0)
    asr = successful_evasions / len(attack_samples)

    print(f"\n3. Attack Results:")
    print(f"   Attack Success Rate (ASR): {asr:.2%}")
    print(f"   Successfully evaded: {successful_evasions} out of {len(attack_samples)}")
    print(f"   Average confidence BEFORE attack: {np.mean(original_confidences):.4f}")
    print(f"   Average confidence AFTER attack: {np.mean(perturbed_confidences):.4f}")
    print(f"   Average confidence drop: {np.mean(original_confidences) - np.mean(perturbed_confidences):.4f}")
else:
    print("\n   No toxic comments found to test attack on.")
    asr = 0

PART 3: ADVERSARIAL ATTACKS (FIXED)
ATTACK 1: CHARACTER-LEVEL EVASION ATTACK

1. Testing attack on sample text:
   Original: I hate you, you're so stupid and ugly
   Perturbed: I hаtе уоu,, ууоuu'rе sо stu​​рi​dd ааnd uglу

2. Finding toxic comments with confidence >= 0.7...


Finding toxic comments: 100%|██████████| 40/40 [01:09<00:00,  1.73s/it]


   Found 971 toxic comments with confidence >= 0.7
   Testing on 500 samples...


Running evasion attack: 100%|██████████| 500/500 [00:04<00:00, 101.81it/s]


3. Attack Results:
   Attack Success Rate (ASR): 99.40%
   Successfully evaded: 497 out of 500
   Average confidence BEFORE attack: 0.9630
   Average confidence AFTER attack: 0.0074
   Average confidence drop: 0.9556


In [5]:
# Attack 2: Label-flipping poisoning attack

from datasets import Dataset
from transformers import Trainer, TrainingArguments
from scipy.special import softmax
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    probs = softmax(logits, axis=-1)[:, 1]
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'auc': roc_auc_score(labels, probs)
    }

print("\n0. Tokenizing evaluation dataset...")
eval_hf = Dataset.from_dict({
    'text': eval_df['comment_text'].tolist(),
    'label': eval_df['toxic_binary'].tolist()
})
eval_tokenized = eval_hf.map(tokenize_function, batched=True, batch_size=1000)
eval_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
print("   Evaluation dataset tokenized")

print("\n1. Creating poisoned training data...")
poisoned_train_df = train_df.copy()
poison_ratio = 0.05
n_poison = int(len(poisoned_train_df) * poison_ratio)

poison_indices = np.random.choice(poisoned_train_df.index, size=n_poison, replace=False)

original_flipped = []
for idx in poison_indices:
    original_label = poisoned_train_df.loc[idx, 'toxic_binary']
    new_label = 1 - original_label
    poisoned_train_df.loc[idx, 'toxic_binary'] = new_label
    original_flipped.append((original_label, new_label))

print(f"   Poisoned {n_poison} rows ({poison_ratio:.0%} of training data)")
print(f"   Label flips: {sum(1 for o,n in original_flipped if o==0)} non-toxic→toxic, {sum(1 for o,n in original_flipped if o==1)} toxic→non-toxic")

print("\n2. Creating poisoned dataset...")
poisoned_dataset = Dataset.from_dict({
    'text': poisoned_train_df['comment_text'].tolist(),
    'label': poisoned_train_df['toxic_binary'].tolist()
})

print("   Tokenizing poisoned dataset...")
poisoned_tokenized = poisoned_dataset.map(tokenize_function, batched=True, batch_size=1000)
poisoned_tokenized.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print("\n3. Training model on poisoned data (this will take ~15-20 minutes)...")

poisoned_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
poisoned_model = poisoned_model.to('cuda')

poisoned_training_args = TrainingArguments(
    output_dir='./poisoned_model',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='no',
    fp16=True,
    report_to='none',
)

poisoned_trainer = Trainer(
    model=poisoned_model,
    args=poisoned_training_args,
    train_dataset=poisoned_tokenized,
    eval_dataset=eval_tokenized,
    compute_metrics=compute_metrics,
)

poisoned_trainer.train()

print("\n4. Evaluating poisoned model...")
poisoned_preds = poisoned_trainer.predict(eval_tokenized)
poisoned_logits = poisoned_preds.predictions
poisoned_probs = softmax(poisoned_logits, axis=-1)[:, 1]
poisoned_preds_binary = np.argmax(poisoned_logits, axis=-1)
poisoned_labels = poisoned_preds.label_ids

poisoned_accuracy = accuracy_score(poisoned_labels, poisoned_preds_binary)
poisoned_f1 = f1_score(poisoned_labels, poisoned_preds_binary, average='macro')
poisoned_precision = precision_score(poisoned_labels, poisoned_preds_binary)
poisoned_recall = recall_score(poisoned_labels, poisoned_preds_binary)
poisoned_fnr = 1 - poisoned_recall  # False Negative Rate

print("\n5. Results comparison:")
print("-" * 70)
print(f"{'Metric':<20} {'Original Model':<20} {'Poisoned Model':<20} {'Change':<15}")
print("-" * 70)

original_accuracy = 0.9456
original_f1 = 0.8130
original_precision = 0.6637
original_recall = 0.6475
original_fnr = 0.3525

print(f"{'Accuracy':<20} {original_accuracy:.4f}{'':<15} {poisoned_accuracy:.4f}{'':<15} {poisoned_accuracy - original_accuracy:+.4f}")
print(f"{'F1 (macro)':<20} {original_f1:.4f}{'':<15} {poisoned_f1:.4f}{'':<15} {poisoned_f1 - original_f1:+.4f}")
print(f"{'Precision':<20} {original_precision:.4f}{'':<15} {poisoned_precision:.4f}{'':<15} {poisoned_precision - original_precision:+.4f}")
print(f"{'Recall':<20} {original_recall:.4f}{'':<15} {poisoned_recall:.4f}{'':<15} {poisoned_recall - original_recall:+.4f}")
print(f"{'False Negative Rate':<20} {original_fnr:.4f}{'':<15} {poisoned_fnr:.4f}{'':<15} {poisoned_fnr - original_fnr:+.4f}")

print("\n6. Key findings:")
print(f"   FNR increased by: {(poisoned_fnr - original_fnr)*100:.2f}%")


0. Tokenizing evaluation dataset...


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

   Evaluation dataset tokenized

1. Creating poisoned training data...
   Poisoned 5000 rows (5% of training data)
   Label flips: 4596 non-toxic→toxic, 404 toxic→non-toxic

2. Creating poisoned dataset...
   Tokenizing poisoned dataset...


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]


3. Training model on poisoned data (this will take ~15-20 minutes)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision,Recall,Auc
1,0.288069,0.160984,0.946850,0.798876,0.715663,0.556875,0.943080
2,0.263729,0.161236,0.947900,0.799791,0.732888,0.548750,0.944656
3,0.233976,0.173346,0.944800,0.808559,0.662304,0.632500,0.934508



4. Evaluating poisoned model...



5. Results comparison:
----------------------------------------------------------------------
Metric               Original Model       Poisoned Model       Change         
----------------------------------------------------------------------
Accuracy             0.9456                0.9448                -0.0008
F1 (macro)           0.8130                0.8086                -0.0044
Precision            0.6637                0.6623                -0.0014
Recall               0.6475                0.6325                -0.0150
False Negative Rate  0.3525                0.3675                +0.0150

6. Key findings:
   FNR increased by: 1.50%


In [7]:
print("\nATTACK COMPARISON:")
print("-" * 60)
print(f"Attack 1 (Evasion) - ASR: {asr:.2%}")
print(f"   • Effort per comment: High (requires per-comment modification)")
print(f"   • Access required: None (can be done by any user)")
print(f"   • Detection difficulty: Medium (perturbations are invisible to humans)")
print(f"   • Impact: Single comment at a time")

print(f"\nAttack 2 (Poisoning) - FNR increase: {(poisoned_fnr - original_fnr)*100:.1f}%")
print(f"   • Effort per comment: Low (once-off data corruption)")
print(f"   • Access required: HIGH (needs training data access)")
print(f"   • Detection difficulty: High (hard to detect during training)")
print(f"   • Impact: Affects ALL future predictions")

print("\n⚠️ OPERATIONAL RISK ASSESSMENT:")
print("-" * 60)

if asr > 0.3:
    print("   Attack 1 (Evasion) is HIGHLY EFFECTIVE (>30% success rate)")
    print("   → Bad actors can easily bypass moderation in real-time")
    print("   → Requires input sanitization and adversarial training")
else:
    print("   Attack 1 (Evasion) has MODERATE effectiveness")
    print("   → Still a concern but not catastrophic")

if (poisoned_fnr - original_fnr) > 0.1:
    print("\n   Attack 2 (Poisoning) is SEVERELY DAMAGING (>10% FNR increase)")
    print("   → Model now misses significantly more toxic content")
    print("   → However, requires training pipeline access (insider threat)")
    print("   → More dangerous but less likely in practice")
else:
    print("\n   Attack 2 (Poisoning) has MODERATE impact")
    print("   → Still concerning but not catastrophic")

print("\nFINAL VERDICT:")
print("-" * 60)
print("For a LIVE PRODUCTION platform:")
print("   → EVASION ATTACKS are more operationally dangerous")
print("   → Reason: Low barrier to entry, real-time execution")
print("   → Any user can test and adapt perturbations instantly")
print("\n   → POISONING ATTACKS are more severe but less realistic")
print("   → Reason: Requires compromised internal systems")
print("   → Still need defenses (data validation, anomaly detection)")

print("\nRECOMMENDED DEFENSES:")
print("   1. Input normalization (remove zero-width chars, normalize Unicode)")
print("   2. Adversarial training (train on perturbed examples)")
print("   3. Ensemble models (reduce single-point vulnerability)")
print("   4. Training data validation (detect label anomalies)")


ATTACK COMPARISON:
------------------------------------------------------------
Attack 1 (Evasion) - ASR: 99.40%
   • Effort per comment: High (requires per-comment modification)
   • Access required: None (can be done by any user)
   • Detection difficulty: Medium (perturbations are invisible to humans)
   • Impact: Single comment at a time

Attack 2 (Poisoning) - FNR increase: 1.5%
   • Effort per comment: Low (once-off data corruption)
   • Access required: HIGH (needs training data access)
   • Detection difficulty: High (hard to detect during training)
   • Impact: Affects ALL future predictions

⚠️ OPERATIONAL RISK ASSESSMENT:
------------------------------------------------------------
   Attack 1 (Evasion) is HIGHLY EFFECTIVE (>30% success rate)
   → Bad actors can easily bypass moderation in real-time
   → Requires input sanitization and adversarial training

   Attack 2 (Poisoning) has MODERATE impact
   → Still concerning but not catastrophic

FINAL VERDICT:
---------------